# 8 pamoka – Daugiavarpis agentų dizaino šablonas


## Diegimas


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kodėl daugiaveiksnių agentų sistemos?

Realių užduočių, kaip kelionės planavimas, atlikimas apima daugybę skirtingų sričių — logistiką, vietines žinias, biudžetavimą ir kt. Vienas agentas, bandantis viską daryti, greitai tampa nevaldomas.

Daugiaveiksnių agentų sistemos tai sprendžia per **specializaciją**: kiekvienas agentas fokusuojasi į vieną sritį, gamindamas kokybiškesnius rezultatus nei generalistas. Jos taip pat pagerina **plečiamumą** — galite pridėti naujų agentų (pvz., skrydžių specialistą, restorano kritiką) nekeisdami esamo darbo proceso. Agentai susijungia per struktūruotą procesą, perduodami kontekstą vienas kitam.


## Specializuotų agentų kūrimas


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sekos darbo eigos kūrimas

`WorkflowBuilder` leidžia sujungti agentus į nukreiptą grafiką. Čia sukuriame paprastą dviejų žingsnių procesą: **TravelPlanner** sudaro kelionės planą, o po to **TravelConcierge** jį peržiūri ir patobulina.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kaip pridėti daugiau agentų į darbo eigą

Vienas didžiausių daugiaagentės konstrukcijos privalumų yra jos išplėtimo paprastumas. Žemiau pridedame **BudgetReviewer** agentą, kuris tikrina planą pagal keliautojo biudžetą, pažymi punktus, kurie gali viršyti limitą, ir siūlo pinigų taupymo alternatyvas. Dabar darbo eiga paeiliui vykdo tris agentus:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Santrauka

Šioje pamokoje jūs išmokote:

1. **Kurti specializuotus agentus** — kiekvienas su susikoncentravusia rolę (planavimas, konsjeržas, biudžeto peržiūra).
2. **Sujungti agentus į seką** naudojant `WorkflowBuilder` ir `add_edge`.
3. **Srautiniu būdu perduoti išvestį** iš kelių agentų darbo eigos, sekant, kuris agentas kalba.
4. **Išplėsti darbo eigą** pridėdami naujų agentų į grandinę nekeisdami esamų.

Kelių agentų dizaino modelis palaiko kiekvieną agentą paprastą, tuo pačiu generuodamas turtingesnius, išsamiau peržiūrėtus rezultatus nei bet kuris vienas agentas galėtų pasiekti vienas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
